In [ ]:
!pip install -U bitsandbytes transformers peft accelerate trl datasets pypdf pandas

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Setup Environment
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

# 1. CONFIGURATION
csv_file = "/content/Dataset.csv"
cv_col = "Context"
roast_col = "Response"
model_id = "unsloth/llama-3-8b-bnb-4bit"
output_dir = "roaster_v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. TOKENIZER & MODEL LOADING
print("⏳ Loading Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# 3. PREPARE FOR QLoRA
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)

# 4. DATASET PREPARATION (With Validation Split)
print("📊 Preparing Dataset...")
df = pd.read_csv(csv_file)
dataset = Dataset.from_pandas(df)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_prompt(example):
    return (
        "### CV Context:\n"
        + str(example[cv_col])
        + "\n\n### Roast:\n"
        + str(example[roast_col])
    )

# 5. TRAINING CONFIGURATION
sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="steps",
    eval_steps=10,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

# 6. TRAIN
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_prompt,
)

print("🚀 Starting Training...")
trainer.train()

# 7. SAVE
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Training Finished & Saved to {output_dir}!")